In [1]:
# Stage 1: Reference ecosystem condition model

# Step 1: Create inputs (46) and outputs (14) for model
# Step 2: Identify relatively intact/natural areas
# Step 3: Build a multi-in and multi-out model
# Step 4: Use environmental covariates (abiotic variables) to predict RS vars indicative of structure and function

In [2]:
# Stage 2: Condition benchmarking

# step 1: Calculate diff between observed and predicted RS vars for all sites.
# step 2: Select relevant sites in reference condition
# step 3: Benchmark condition by comparing diff values to diff values at reference sites. produces uncalibrated HCI
# Step 4: Calibrate HCI to 0-1 scale using reference site diff values

In [3]:
# Suggest alternative datasets for HCAS in SA, Africa, or globally. Identify missing datasets.
# Implement HCAS for SA.
# Make python code publicly available on GitHub.

In [4]:
# use conformal quantile regression to generate prediction intervals for RS variables
# based on intervals, select reference sites that fall in this range.

# use BII to select reference sites, after combining with multiple datasets of natural areas. 
# Select BII threshold that best splits natural and transformed areas.

In [6]:
import ee

ee.Authenticate(auth_mode='notebook')
ee.Initialize(project = 'ee-gsingh')

import geemap
import eemont


c:\Users\coach\.conda\envs\erthy\Lib\site-packages\ee_extra\JavaScript\translate_functions.py:110: SyntaxWarning: invalid escape sequence '\.'
  tentative_name = regex.findall("(.*\..*)\s=\sfunction\(", fn_header)
c:\Users\coach\.conda\envs\erthy\Lib\site-packages\ee_extra\JavaScript\translate_functions.py:144: SyntaxWarning: invalid escape sequence '\s'
  init_space = regex.match("\s*", fn_header)[0]
c:\Users\coach\.conda\envs\erthy\Lib\site-packages\ee_extra\JavaScript\translate_functions.py:392: SyntaxWarning: invalid escape sequence '\s'
  init_space = regex.match("\s*", fn_header)[0]
c:\Users\coach\.conda\envs\erthy\Lib\site-packages\ee_extra\JavaScript\translate_functions.py:602: SyntaxWarning: invalid escape sequence '\s'
  if bool(regex.search("=\s*function", line)):
c:\Users\coach\.conda\envs\erthy\Lib\site-packages\ee_extra\JavaScript\translate_jsm_wrappers.py:45: SyntaxWarning: invalid escape sequence '\w'
  fcondition = "([\w'\"\]\)%s]+?)\.%s\((?>[^()]+|(?1))*\)" % (extra, 

In [7]:
# natural lands from SBTN from SBTN
sbtn = ee.Image("WRI/SBTN/naturalLands/v1_1/2020")

# BII (Clements et al., 2025)
bii1km = ee.ImageCollection("projects/earthengine-legacy/assets/projects/sat-io/open-datasets/BII/BII_1km")
mask = ee.Image("projects/earthengine-legacy/assets/projects/sat-io/open-datasets/BII/BII_Mask")

bands1km = ee.List(['Land Use', 'Land Use Intensity', 'BII All',
'BII Amphibians', 'BII Birds', 'BII Forbs', 'BII Graminoids',
'BII Mammals', 'BII All Plants', 'BII Reptiles', 'BII Trees',
'BII All Vertebrates'])

bands8km = ee.List(['BII All',
'BII Amphibians', 'BII Birds', 'BII Forbs', 'BII Graminoids',
'BII Mammals', 'BII All Plants', 'BII Reptiles', 'BII Trees',
'BII All Vertebrates', 'Land Use', 'Land Use Intensity'])

# Process 1km dataset
bii1km = bii1km.toBands().rename(bands1km)
biionekm = bii1km.select('^BII.*').selfMask()
lcMask1km = bii1km.select('Land Use').neq(2).And(bii1km.select('Land Use').neq(5))
LUI1km = bii1km.select('Land Use Intensity').updateMask(lcMask1km)
bii1km_processed = biionekm.addBands([bii1km.select('Land Use'), LUI1km]).updateMask(mask)

# GHM v3
HM_90M = ee.ImageCollection("projects/sat-io/open-datasets/GHM/HM_2022_90M")

# Natural forests data
probabilities = ee.ImageCollection(
        'projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection')\
        .mosaic()\
        .select('B0')

In [8]:
bii1km_processed.bandNames()

In [4]:
Map = geemap.Map()
Map.addLayer(bii1km_processed, {}, 'BII 1km processed')
Map.addLayer(sbtn, {}, 'SBTN Natural Lands')
Map.addLayer(probabilities.mask(probabilities.neq(0)),
    {min: 0, max: 250, 'palette': ['white', 'green']},
    'Natural forest probabilities')
Map.addLayer(
    probabilities.gte(0.52).mask(probabilities.gte(0.5)), {'palette': 'teal'},
    'Natural forest map at threshold')
# Map.addLayer(point, {'color': 'red'}, 'Point')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [9]:
# import lsib and filter to south africa
sa = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'South Africa'))

# Ecoregions
ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(sa.geometry())

econames = ecoregions.aggregate_array('ECO_NAME')
econames

listid = 6
econame = ecoregions.aggregate_array('ECO_NAME').get(listid).getInfo()
print(f"Selecting {econame} ecoregion")
ecoregion = ecoregions.filter(ee.Filter.eq('ECO_NAME', econame))

Selecting Highveld grasslands ecoregion


In [10]:
sbtn_nat = sbtn.select('natural')
ghm_nat = ee.Image(HM_90M.select(0).filterBounds(ecoregion.geometry()).first()).lte(0.1).rename('natural')
bii_nat = bii1km_processed.select('BII All').gte(0.7).rename('natural')
nf_nat = probabilities.gte(0.52).rename('natural')

natural_mask = sbtn_nat.Or(nf_nat).And(ghm_nat).And(bii_nat).selfMask().rename('natural')

In [43]:
Map = geemap.Map()
# Map.addLayer(sbtn_nat, {}, 'SBTN Natural Lands')
# Map.addLayer(ghm_nat, {}, 'GHM Natural Lands')
# Map.addLayer(bii_nat, {}, 'BII Natural Lands')
# Map.addLayer(nf_nat, {}, 'Natural Forest Natural Lands')
Map.addLayer(ecoregion.geometry(), {}, 'Ecoregion')
Map.addLayer(natural_mask, {'palette': 'green'}, 'Combined Natural Lands Mask')
Map.centerObject(ecoregion, 8)
Map

Map(center=[-27.791878438079546, 27.763014210969725], controls=(WidgetControl(options=['position', 'transparen…

In [5]:
# Step 1: create a near-natural layer
# grasslands WRI
# invasive species data

In [6]:
# embeddings predict indices
# Filter for year, cloud mask, compute median composites
# kndvi
# nbr
# ndmi
# ndwi
# bsi

# sdev
# edev
# bcdev
# pv10
# pvrange
# npv10
# npvrange
# bs10
# bsrange

In [ ]:
point = ee.Geometry.PointFromQuery(
    'South Africa',
    user_agent = 'eemont-example'
) # Extended constructor

S2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2022-01-01', '2023-01-01')
    .filterBounds(ecoregion.geometry().centroid(10))
    # .closest('2020-10-15') # Extended (pre-processing)
    .maskClouds(prob = 60) # Extended (pre-processing)
    .scaleAndOffset() # Extended (pre-processing)
    .spectralIndices(['NBR', 'NDMI', 'NDWI']) # Extended (processing)
    .select(['NBR', 'NDMI', 'NDWI']).median()).rename(['NBR', 'NDMI', 'NDWI'])
S2
Map = geemap.Map()
Map.addLayer(ecoregion.geometry().centroid(5))
Map.addLayer(S2)
Map.centerObject(ecoregion.geometry().centroid(5), 10)
Map

Map(center=[-27.79187843807928, 27.763014210968926], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
fscs = ee.FeatureCollection("projects/ee-gsingh/assets/RECOVER/fscs_aef_samples").filterBounds(ecoregion.geometry()).limit(5)
fList = fscs.toList(5)
processedIds = []
fscsfiltered = fscs.filter(ee.Filter.inList('system:index', processedIds).Not())
for i in range(5):
    ft = ee.Feature(fList.get(i))
    s2Imagebbox = s2Collection.filterBounds(ft.geometry()).first()
    pts = fscsfiltered.filterBounds(s2Imagebbox.geometry())
    if pts.size().getInfo() > 0:
        pts = pts.map(lambda ft: ft.set('id', ft.get('system:index')))

        eedf = s2Collection.reduceRegions(pts)
        df = ee.data.computeFeatures({'expression': eedf, 'format': 'PANDAS DATAFRAME'})
        processedIds.append(df['id'])
        # write df to csv in append mode
        df.to_csv("test_extracted_indices.csv", mode='a', header=False)
    else:
        print("No points found for this image.")

    

In [57]:
# --- Define Export Parameters ---
asset_id = 'projects/ee-gsingh/assets/RECOVER/covs22_indices'  # **IMPORTANT: Change this path**
scale_resolution = 10  # Sentinel-2 native resolution is 10 meters

# --- Start the Export Task ---
task = ee.batch.Export.image.toAsset(
    image=S2,
    description='S2_Indices_Median_Export', # A human-readable name for the task
    assetId=asset_id,
    scale=scale_resolution,
    region=ecoregion.geometry().bounds(), # Use the bounds of your region for the export extent
    maxPixels=1e13 # Increase maxPixels for larger areas
)

# Start the task
task.start()

# Print the task ID and status
print(f"Export task started. Asset ID: {asset_id}")
print(f"Check task status: {task.status()['state']}")

Export task started. Asset ID: projects/ee-gsingh/assets/RECOVER/covs22_indices
Check task status: READY


In [49]:
from geeml.utils import eeprint

In [51]:
fscs = ee.FeatureCollection("projects/ee-gsingh/assets/RECOVER/fscs_aef_samples").filterBounds(ecoregion.geometry())
train = natural_mask.unmask(0).reduceRegions(collection = fscs, reducer = ee.Reducer.first(), scale = natural_mask.projection().nominalScale())
train_filtered = train.filter(ee.Filter.eq('first', 1)).limit(5)
eedf = S2.reduceRegions(collection = train_filtered, reducer = ee.Reducer.first(), scale = S2.projection().nominalScale())
    # .filter(ee.Filter.notNull(['first', 'compositio']))
eedf.first()

## Expected reference model

In [3]:
import pandas as pd

df = pd.read_csv(r"C:\Users\coach\myfiles\postdoc2\code\extracted_indices.csv")
df.head()

,geo,A00,A01,A02,A03,A04,A05,A06,A07,A08,...,A59,A60,A61,A62,A63,NBR,NDMI,NDWI,cell_index,id
0,"{'type': 'Point', 'coordinates': [25.558372199...",0.147697,0.051734,-0.135886,-0.032541,-0.032541,-0.147697,0.044844,0.259900,-0.003937,...,-0.059116,0.051734,0.008858,-0.051734,-0.147697,0.316783,0.088858,-0.615630,6,00000000000000000164
1,"{'type': 'Point', 'coordinates': [25.577417019...",0.119093,0.147697,-0.041584,0.048228,-0.071111,-0.108512,0.066990,0.221453,-0.003014,...,0.017778,0.029773,0.147697,-0.108512,-0.124567,0.166577,-0.040355,-0.490129,6,00000000000000000172
2,"{'type': 'Point', 'coordinates': [25.551366951...",0.119093,0.035433,-0.079723,-0.029773,-0.032541,-0.108512,0.032541,0.221453,0.000000,...,0.012057,0.071111,0.079723,-0.066990,-0.160000,0.232205,0.018989,-0.528391,6,0000000000000000012d
3,"{'type': 'Point', 'coordinates': [25.578852850...",0.172795,0.119093,-0.093564,0.006151,-0.093564,-0.088827,0.062991,0.221453,0.022207,...,-0.007443,0.041584,0.088827,-0.032541,-0.119093,0.133381,-0.062832,-0.491525,6,0000000000000000017a
4,"{'type': 'Point', 'coordinates': [25.567357287...",0.093564,0.079723,-0.041584,0.012057,-0.088827,-0.166336,0.066990,0.214133,-0.000246,...,0.041584,0.022207,0.124567,-0.124567,-0.199862,0.148976,-0.039301,-0.490157,6,0000000000000000016a


In [ ]:
Xcols = ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08',
       'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18',
       'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28',
       'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38',
       'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48',
       'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58',
       'A59', 'A60', 'A61', 'A62', 'A63']
ycols = ['NBR', 'NDMI', 'NDWI']
df.shape
df.columns

Index(['geo', 'A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08',
       'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18',
       'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28',
       'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38',
       'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48',
       'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58',
       'A59', 'A60', 'A61', 'A62', 'A63', 'NBR', 'NDMI', 'NDWI', 'cell_index',
       'id'],
      dtype='object')